In [1]:
!pip install boto3

In [ ]:
!pip -q install json_repair

In [2]:
# Copyright Amazon.com, Inc. or its affiliates. All Rights Reserved.
# SPDX-License-Identifier: Apache-2.0
"""
Shows how to use the Converse API with Anthropic Claude 3 Sonnet (on demand).
"""



import logging
import boto3
import json
import time
import random
from botocore.exceptions import ClientError
from botocore.config import Config

config = Config(read_timeout=1000)


logger = logging.getLogger(__name__)
logging.basicConfig(level=logging.ERROR)


bedrock_client = boto3.client(service_name='bedrock-runtime', 
                            region_name='XXX', 
                            aws_access_key_id='XXX',  
                            aws_secret_access_key='XXX', 
                            config=config)


def complete_chat(prompt=None,
                img_path=None,
                #model_id="anthropic.claude-3-5-sonnet-20241022-v2:0",
                model_id = "us.anthropic.claude-sonnet-4-20250514-v1:0",
                system_prompt="You are a helpful assistant", 
                toolConfig=None, 
                temperature=None,
                maxTokens=8192,
                messages=None,
                extended_thinking=False
                ):
    """
    Sends messages to a model.
    Args:
        bedrock_client: The Boto3 Bedrock runtime client.
        model_id (str): The model ID to use.
        system_prompts (JSON) : The system prompts for the model to use.
        messages (JSON) : The messages to send to the model.

    Returns:
        response (JSON): The conversation that the model generated.

    """
    #TODO: Drive these through env variables when migration is complete
    
    seconds = random.sample([1, 3, 5], k=1)[0]
    time.sleep(seconds)

    
    logger.info("Generating message with model %s", model_id)

    # Inference parameters to use.
    if not temperature:
        temperature = 0
    top_k = 500
    #maxTokens = 4096

    # Additional inference parameters to use.

    additional_model_fields = {"top_k": top_k}
    if extended_thinking:
        if not temperature:
            temperature = 1
        additional_model_fields = {
            "reasoning_config": {
                "type": "enabled",
                "budget_tokens": 1024
            }}

    # Base inference parameters to use.
    inference_config = {"temperature": temperature, "maxTokens": maxTokens}

    if img_path:
        with open(img_path, "rb") as f:
            image = f.read()


        messages = [
    {
        "role": "user",
        "content": [
            {"image": {"format": "png", "source": {"bytes": image}}},
            {"text": prompt},
        ],
    }
]
    elif prompt:
        messages = [{
            "role": "user",
            "content": [{"text": prompt}]}]

        
    system_prompts = [{"text": system_prompt}]

    # Send the message.
    if toolConfig:
        response = bedrock_client.converse(
            modelId=model_id,
            messages=messages,
            system=system_prompts,
            inferenceConfig=inference_config,
            additionalModelRequestFields=additional_model_fields, 
            toolConfig=toolConfig
        )
    else:
        response = bedrock_client.converse(
            modelId=model_id,
            messages=messages,
            system=system_prompts,
            inferenceConfig=inference_config,
            additionalModelRequestFields=additional_model_fields
        )

    # Log token usage.
    #token_usage = response['.localized']
    #logger.info("Input tokens: %s", token_usage['inputTokens'])
    #logger.info("Output tokens: %s", token_usage['outputTokens'])
    #logger.info("Total tokens: %s", token_usage['totalTokens'])
    #logger.info("Stop reason: %s", response['stopReason'])

    if toolConfig:
        response_message = response['output']['message']
        response_content_blocks = response_message['content']
        content_block = next((block for block in response_content_blocks if 'toolUse' in block), None)
        tool_use_block = content_block['toolUse']
        tool_result_dict = tool_use_block['input']

        return json.dumps(tool_result_dict, indent=4)
 
    else:
        if extended_thinking:
            return response["output"]["message"]["content"][1]["text"]
            
        return response["output"]["message"]["content"][0]["text"]
 

In [3]:
complete_chat('who is narendra modi')

"Narendra Modi is the current Prime Minister of India, having held the position since May 2014. Here are some key facts about him:\n\n**Political Background:**\n- Leader of the Bharatiya Janata Party (BJP)\n- Previously served as Chief Minister of Gujarat state from 2001 to 2014\n- Won re-election as Prime Minister in 2019\n\n**Early Life:**\n- Born on September 17, 1950, in Vadnagar, Gujarat\n- Comes from a modest background; his family ran a tea stall\n- Joined the Rashtriya Swayamsevak Sangh (RSS) as a young man\n\n**Key Policies and Initiatives:**\n- Digital India campaign\n- Make in India manufacturing initiative\n- Swachh Bharat (Clean India) mission\n- Goods and Services Tax (GST) implementation\n- Demonetization policy in 2016\n\n**International Profile:**\n- Active on the global stage with frequent international visits\n- Promotes India's economic growth and strategic partnerships\n- Known for his use of social media and public communication\n\nModi is a significant and often 

In [4]:
tactical_patterns = '''[
  {
    "motif": "Advanced pawn",
    "motif_description": "One of your pawns is deep into the opponent position, maybe threatening to promote."
  },
  {
    "motif": "Attacking f2 or f7",
    "motif_description": "An attack focusing on the f2 or f7 pawn, such as in the fried liver opening."
  },
  {
    "motif": "Capture the defender",
    "motif_description": "Removing a piece that is critical to defense of another piece, allowing the now undefended piece to be captured on a following move."
  },
  {
    "motif": "Discovered attack",
    "motif_description": "Moving a piece that previously blocked an attack by another long range piece, such as a knight out of the way of a rook."
  },
  {
    "motif": "Double check",
    "motif_description": "Checking with two pieces at once, as a result of a discovered attack where both the moving piece and the unveiled piece attack the opponent's king."
  },
  {
    "motif": "Exposed king",
    "motif_description": "A tactic involving a king with few defenders around it, often leading to checkmate."
  },
  {
    "motif": "Fork",
    "motif_description": "A move where the moved piece attacks two opponent pieces at once."
  },
  {
    "motif": "Hanging piece",
    "motif_description": "A tactic involving an opponent piece being undefended or insufficiently defended and free to capture."
  },
  {
    "motif": "Kingside attack",
    "motif_description": "An attack of the opponent's king, after they castled on the king side."
  },
  {
    "motif": "Pin",
    "motif_description": "A tactic involving pins, where a piece is unable to move without revealing an attack on a higher value piece."
  },
  {
    "motif": "Queenside attack",
    "motif_description": "An attack of the opponent's king, after they castled on the queen side."
  },
  {
    "motif": "Sacrifice",
    "motif_description": "A tactic involving giving up material in the short-term, to gain an advantage again after a forced sequence of moves."
  },
  {
    "motif": "Skewer",
    "motif_description": "A motif involving a high value piece being attacked, moving out the way, and allowing a lower value piece behind it to be captured or attacked, the inverse of a pin."
  },
  {
    "motif": "Trapped piece",
    "motif_description": "A piece is unable to escape capture as it has limited moves."
  },
  {
    "motif": "Attraction",
    "motif_description": "An exchange or sacrifice encouraging or forcing an opponent piece to a square that allows a follow-up tactic."
  },
  {
    "motif": "Clearance",
    "motif_description": "A move, often with tempo, that clears a square, file or diagonal for a follow-up tactical idea."
  },
  {
    "motif": "Defensive move",
    "motif_description": "A precise move or sequence of moves that is needed to avoid losing material or another advantage."
  },
  {
    "motif": "Deflection",
    "motif_description": "A move that distracts an opponent piece from another duty that it performs, such as guarding a key square. Sometimes also called \"overloading\"."
  },
  {
    "motif": "Interference",
    "motif_description": "Moving a piece between two opponent pieces to leave one or both opponent pieces undefended, such as a knight on a defended square between two rooks."
  },
  {
    "motif": "Intermezzo",
    "motif_description": "Instead of playing the expected move, first interpose another move posing an immediate threat that the opponent must answer. Also known as \"Zwischenzug\" or \"In between\"."
  },
  {
    "motif": "Quiet move",
    "motif_description": "A move that does not make a check or capture, but does prepare an unavoidable threat for a later move."
  },
  {
    "motif": "X-Ray attack",
    "motif_description": "A piece attacks or defends a square, through an enemy piece."
  },
  {
    "motif": "Zugzwang",
    "motif_description": "The opponent is limited in the moves they can make, and all moves worsen their position."
  },
  {
    "motif": "Checkmate",
    "motif_description": "Win the game with style."
  },
  {
    "motif": "Mate in 1",
    "motif_description": "Deliver checkmate in one move."
  },
  {
    "motif": "Mate in 2",
    "motif_description": "Deliver checkmate in two moves."
  },
  {
    "motif": "Mate in 3",
    "motif_description": "Deliver checkmate in three moves."
  },
  {
    "motif": "Mate in 4",
    "motif_description": "Deliver checkmate in four moves."
  },
  {
    "motif": "Mate in 5 or more",
    "motif_description": "Figure out a long mating sequence."
  },
  {
    "motif": "Anastasia's mate",
    "motif_description": "A knight and rook or queen team up to trap the opposing king between the side of the board and a friendly piece."
  },
  {
    "motif": "Arabian mate",
    "motif_description": "A knight and a rook team up to trap the opposing king on a corner of the board."
  },
  {
    "motif": "Back rank mate",
    "motif_description": "Checkmate the king on the home rank, when it is trapped there by its own pieces."
  },
  {
    "motif": "Boden's mate",
    "motif_description": "Two attacking bishops on criss-crossing diagonals deliver mate to a king obstructed by friendly pieces."
  },
  {
    "motif": "Double bishop mate",
    "motif_description": "Two attacking bishops on adjacent diagonals deliver mate to a king obstructed by friendly pieces."
  },
  {
    "motif": "Dovetail mate",
    "motif_description": "A queen delivers mate to an adjacent king, whose only two escape squares are obstructed by friendly pieces."
  },
  {
    "motif": "Hook mate",
    "motif_description": "Checkmate with a rook, knight, and pawn along with one enemy pawn to limit the enemy king's escape."
  },
  {
    "motif": "Kill box mate",
    "motif_description": "A rook is next to the enemy king and supported by a queen that also blocks the king's escape squares. The rook and the queen catch the enemy king in a 3 by 3 \"kill box\"."
  },
  {
    "motif": "Vukovic mate",
    "motif_description": "A rook and knight team up to mate the king. The rook delivers mate while supported by a third piece, and the knight is used to block the king's escape squares."
  },
  {
    "motif": "Smothered mate",
    "motif_description": "A checkmate delivered by a knight in which the mated king is unable to move because it is surrounded (or smothered) by its own pieces."
  },
  {
    "motif": "Castling",
    "motif_description": "Bring the king to safety, and deploy the rook for attack."
  },
  {
    "motif": "En passant",
    "motif_description": "A tactic involving the en passant rule, where a pawn can capture an opponent pawn that has bypassed it using its initial two-square move."
  },
  {
    "motif": "Promotion",
    "motif_description": "Promote one of your pawns to a queen or minor piece."
  },
  {
    "motif": "Underpromotion",
    "motif_description": "Promotion to a knight, bishop, or rook."
  },
  {
    "motif": "Equality",
    "motif_description": "Come back from a losing position, and secure a draw or a balanced position. (eval ≤ 200cp)"
  },
  {
    "motif": "Advantage",
    "motif_description": "Seize your chance to get a decisive advantage. (200cp ≤ eval ≤ 600cp)"
  },
  {
    "motif": "Crushing",
    "motif_description": "Spot the opponent blunder to obtain a crushing advantage. (eval ≥ 600cp)"
  },
  {
    "motif": "Checkmate",
    "motif_description": "Win the game with style."
  },
  {
    "motif": "One-move puzzle",
    "motif_description": "A puzzle that is only one move long."
  },
  {
    "motif": "Short puzzle",
    "motif_description": "Two moves to win."
  },
  {
    "motif": "Long puzzle",
    "motif_description": "Three moves to win."
  },
  {
    "motif": "Very long puzzle",
    "motif_description": "Four moves or more to win."
  },
  {
    "motif": "Master games",
    "motif_description": "Puzzles from games played by titled players."
  },
  {
    "motif": "Master vs Master games",
    "motif_description": "Puzzles from games between two titled players."
  },
  {
    "motif": "Super GM games",
    "motif_description": "Puzzles from games played by the best players in the world."
  },
  {
    "motif": "Player games",
    "motif_description": "Lookup puzzles generated from your games, or from another player's games."
  }
]
'''

In [5]:
import json
with open('usernam12_detailed_analysis.json') as f:
    data = json.load(f)

In [6]:
fens = []
for i in range(len(data)):
    for log in data[i]['detailed_move_logs']:
        if log['move_quality']=='mistake':
            if log['eval_after_actual_move']>-3:
                fens.append(log)
            

In [9]:
fens[0]

{'move_number': 3,
 'position_fen': 'rnbqkbnr/ppp2ppp/3p4/4p3/4P3/2N5/PPPP1PPP/R1BQKBNR w KQkq - 0 3',
 'actual_move_played': 'f4',
 'best_move': 'd4',
 'best_move_line': 'd4 Nf6 Nf3 Nbd7 Be3',
 'eval_after_actual_move': 0.1,
 'eval_after_best_move': -0.49,
 'actual_move_inferior_line': 'f4 exf4 Qf3 Nc6 Qxf4 Nd4',
 'eval_difference': 0.59,
 'is_player_move': True,
 'move_quality': 'mistake'}

In [8]:
len(fens)

302

In [10]:
prompt = '''
Here is a chess position analysis: '''+str(fens[54])+'''
Which tactical pattern would the position fall into.
here is a list of possible tactical patterns:
'''+ tactical_patterns+'''
Which tactical pattern would the position fall into.
Further, classify the game state into 'opening','middle game','endgame'
Please give your answer as a JSON with the keys as possible_motifs, reasoning, game_state.
Note that the scores are calculated in such a way that -ve value is advantageous for white and positive is adv for black.
If black has a threatening move that white dodges by playing that in best move line - please give the reasoning with the threat from black (similarly for white)
In a similar way, if white is winning in the best line and not in the actual played line - give reasoning fo what white missed'''

In [11]:
t = complete_chat(prompt)

In [12]:
t

'Looking at this chess position, I need to analyze the tactical patterns and game state.\n\n**Position Analysis:**\n- Move 33 in the game indicates we\'re in the endgame\n- Material appears roughly equal with pawns, knights, and bishops remaining\n- White\'s king is on g1, Black\'s king is on d6\n- Black has a knight on e3 that\'s quite active\n- White played Kf2 instead of the better Nd2\n\n**Tactical Pattern Analysis:**\nThe key tactical element here is that Black\'s knight on e3 is very actively placed and threatening. In the actual line after Kf2, Black plays Nd1+ with a fork, attacking both the king and the b2 pawn. The best move Nd2 would have better coordinated White\'s pieces and avoided this tactical shot.\n\n```json\n{\n  "possible_motifs": [\n    "Fork",\n    "Defensive move", \n    "Exposed king",\n    "Advantage"\n  ],\n  "reasoning": "The position features a fork motif where Black\'s knight on e3 can play Nd1+ after White\'s Kf2, forking the king and b2 pawn. White\'s act

In [13]:
fens[54]

{'move_number': 33,
 'position_fen': '8/1p4p1/p2kpp2/5b2/8/4nNPp/PP5P/2N3K1 w - - 5 33',
 'actual_move_played': 'Kf2',
 'best_move': 'Nd2',
 'best_move_line': 'Nd2 e5 Ne2 Kc6 Nc3',
 'eval_after_actual_move': 5.29,
 'eval_after_best_move': 4.72,
 'actual_move_inferior_line': 'Kf2 Nd1+ Ke1 Nxb2 Ne2 Nc4',
 'eval_difference': 0.57,
 'is_player_move': True,
 'move_quality': 'mistake'}

In [14]:
def get_prompt(fen):
    return '''
Here is a chess position analysis: '''+str(fen)+'''
Which tactical pattern would the position fall into.
here is a list of possible tactical patterns:
'''+ tactical_patterns+'''
Which tactical pattern would the position fall into.
Further, classify the game state into 'opening','middle game','endgame'
Please give your answer as a JSON with the keys as possible_motifs, reasoning, game_state.
Note that the scores are calculated in such a way that -ve value is advantageous for white and positive is adv for black.
If black has a threatening move that white dodges by playing that in best move line - please give the reasoning with the threat from black (similarly for white)
In a similar way, if white is winning in the best line and not in the actual played line - give reasoning fo what white missed'''

In [15]:
prompt = get_prompt(fens[0])

In [16]:
t = complete_chat(prompt)

In [81]:
def get_response(fen):
    prompt = get_prompt(fen)
    t = complete_chat(prompt)
    return fen, t

In [86]:
from concurrent.futures import ProcessPoolExecutor, as_completed
from tqdm import tqdm

def process_fens_parallel(fens_list, max_workers=10):
    results = []
    
    with ProcessPoolExecutor(max_workers=max_workers) as executor:
        future_to_fen = {executor.submit(get_response, fen): fen for fen in fens_list}
        
        # Create progress bar
        with tqdm(total=len(fens_list), desc="Processing FENs") as pbar:
            for future in as_completed(future_to_fen):
                try:
                    fen, response = future.result()
                    results.append((fen, response))
                except Exception as e:
                    fen = future_to_fen[future]
                    print(f"Error processing {fen}: {e}")
                    results.append((fen, None))
                
                # Update progress bar
                pbar.update(1)
    
    return results

In [ ]:
x = process_fens_parallel(fens[:100])

Processing FENs:  16%|█▌        | 16/100 [00:26<01:36,  1.14s/it]

In [95]:
len(x)

100

In [102]:
x

[({'move_number': 4,
   'position_fen': 'rnbqkbnr/ppp2ppp/3p4/8/4Pp2/2N5/PPPP2PP/R1BQKBNR w KQkq - 0 4',
   'actual_move_played': 'Nf3',
   'best_move': 'Qf3',
   'best_move_line': 'Qf3 Nc6 Qxf4 Nd4 Bd3',
   'eval_after_actual_move': 0.65,
   'eval_after_best_move': 0.0,
   'actual_move_inferior_line': 'Nf3 g5 d4 g4 Ng1 Qh4+',
   'eval_difference': 0.65,
   'is_player_move': True,
   'move_quality': 'mistake'},
  'Looking at this chess position, I need to analyze the tactical patterns and game state.\n\nFrom the FEN position `rnbqkbnr/ppp2ppp/3p4/8/4Pp2/2N5/PPPP2PP/R1BQKBNR w KQkq - 0 4`, I can see:\n- Black has played f5-f4, creating an advanced pawn on f4\n- White has a knight on c3\n- The best move is Qf3 (attacking the f4 pawn), but White played Nf3 instead\n- This allowed Black to continue with g5, creating a pawn storm\n\nThe evaluation shows White went from equal (0.0 after Qf3) to slightly worse (0.65 after Nf3), indicating White missed the opportunity to immediately challenge 

In [99]:
jsons = []
for i in range(len(x)):
    jsons.append(json_repair.loads(x[i][1]))

In [100]:
len(jsons)

100

In [101]:
jsons[0]

{'possible_motifs': ['Advanced pawn',
  'Kingside attack',
  'Defensive move',
  'Advantage'],
 'reasoning': "The position features Black's advanced f4 pawn as the central tactical element. The best move Qf3 would have been a defensive move to immediately challenge this advanced pawn and maintain equality. By playing Nf3 instead, White allowed Black to continue the kingside pawn storm with g5, giving Black the initiative for a kingside attack. White missed the chance to neutralize Black's advanced pawn structure and allowed Black to gain an advantage through the pawn advance.",
 'game_state': 'opening'}

In [103]:
with open('output.txt', 'w') as file:
    for item in x:
        file.write(f"{item}\n")

In [107]:
with open('output.txt') as f:
    t = f.readlines()

In [110]:
import ast
ast.literal_eval(t[0])

({'move_number': 4,
  'position_fen': 'rnbqkbnr/ppp2ppp/3p4/8/4Pp2/2N5/PPPP2PP/R1BQKBNR w KQkq - 0 4',
  'actual_move_played': 'Nf3',
  'best_move': 'Qf3',
  'best_move_line': 'Qf3 Nc6 Qxf4 Nd4 Bd3',
  'eval_after_actual_move': 0.65,
  'eval_after_best_move': 0.0,
  'actual_move_inferior_line': 'Nf3 g5 d4 g4 Ng1 Qh4+',
  'eval_difference': 0.65,
  'is_player_move': True,
  'move_quality': 'mistake'},
 'Looking at this chess position, I need to analyze the tactical patterns and game state.\n\nFrom the FEN position `rnbqkbnr/ppp2ppp/3p4/8/4Pp2/2N5/PPPP2PP/R1BQKBNR w KQkq - 0 4`, I can see:\n- Black has played f5-f4, creating an advanced pawn on f4\n- White has a knight on c3\n- The best move is Qf3 (attacking the f4 pawn), but White played Nf3 instead\n- This allowed Black to continue with g5, creating a pawn storm\n\nThe evaluation shows White went from equal (0.0 after Qf3) to slightly worse (0.65 after Nf3), indicating White missed the opportunity to immediately challenge Black\'s paw